In [30]:
import json
from pyexpat.errors import messages

import openai
from openai.types.chat import ChatCompletionMessage
import requests
client = openai.OpenAI()
api_url = 'https://nomad-movies.nomadcoders.workers.dev'
messages = []


# API 기본 URL: https://nomad-movies.nomadcoders.workers.dev
# 과제1.
#
# 사용자의 취향을 기억하는 영화 추천 챗봇을 구축하세요.
# 챗봇이 갖춰야 할 기능:
# 사용자가 좋아하는 장르를 기억
# 이미 시청한 영화를 기억
# 대화 기록을 기반으로 개인화된 추천 제공
# 요구사항
#
# 대화 기록을 저장하기 위한 messages 리스트를 구현하세요.
# 사용자 메시지(role: "user")와 AI 응답(role: "assistant")을 모두 append하세요.
# 이미 본 영화는 추천하지 않도록 메모리를 활용해야 합니다.
# 챗봇 테스트
#
# Jupyter Notebook에서 다음과 같은 대화를 보여주세요:
# User: 나는 SF 영화를 좋아해
# AI: 좋은 취향이시네요! SF에는 명작이 정말 많죠...
# User: 인셉션이랑 인터스텔라는 이미 봤어
# AI: 좋은 선택이셨네요! 이미 보셨으니까...
# User: 오늘 밤에 뭐 볼지 추천해 줄래?
# AI: SF를 좋아하시고, 인셉션과 인터스텔라는 이미 보셨으니까 추천드리자면...
# User: 내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?
# AI: SF를 좋아하신다고 했죠! 인셉션과 인터스텔라를 보셨습니다.

def get_popular_movies():
    response = requests.get(f'{api_url}/movies')
    response.raise_for_status()
    return response.text  # 또는 response.json()

def get_movie_details(id):
    response = requests.get(f'{api_url}/movies/{id}')
    response.raise_for_status()
    return response.text

def get_movie_credits(id):
    response = requests.get(f'{api_url}/movies/{id}/credits')
    response.raise_for_status()
    return response.text

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of popular movies"
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get a movie's details by ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "description": "Movie ID",
                        "type": "integer"
                    }
                },
                "required": ["id"]
            }
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get a movie's cast and crew by ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "description": "Movie ID",
                        "type": "integer"
                    }
                },
                "required": ["id"]
            }
        }
    }
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello!"}],
    tools=TOOLS
)

def process_ai_message(message: ChatCompletionMessage):
    if message.tool_calls is not None:
        messages.append({
            "role": "assistant",
            "content": f"Calling function: {message.content or ""}",
            "tool_calls": [{
                "id": tool_call.id,
                "type": "function",
                "function": {
                    "name" : tool_call.function.name,
                    "arguments": tool_call.function.arguments,
                }
            } for tool_call in message.tool_calls]
        })
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with arguments: {arguments}")
            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_call = FUNCTION_MAP[function_name]

            if arguments == {}:
                result = function_to_call()
            else:
                result = function_to_call(**arguments)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result
            })

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS
    )
    message = response.choices[0].message.content
    process_ai_message(response.choices[0].message)

    messages.append({"role": "assistant", "content": message})
    print(f"AI: {message}")


while True:
    message = input("Send a message to the LLM")
    if message == "exit" or message == "quit" or message == "bye":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"You: {messages[-1]['content']}")
        call_ai()



You: 나는 SF 영화를 좋아해
Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='SF 영화를 좋아하신다니 좋네요! SF 영화는 상상력이 풍부하고 미래의 기술이나 우주 탐사와 같은 다양한 주제를 다루기 때문에 많은 사람들이 즐기는 장르입니다. 어떤 특정한 SF 영화가 궁금하신가요? 아니면 인기 있는 SF 영화를 추천해 드릴까요?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))
AI: SF 영화를 좋아하신다니 좋네요! SF 영화는 상상력이 풍부하고 미래의 기술이나 우주 탐사와 같은 다양한 주제를 다루기 때문에 많은 사람들이 즐기는 장르입니다. 어떤 특정한 SF 영화가 궁금하신가요? 아니면 인기 있는 SF 영화를 추천해 드릴까요?
AI: SF 영화를 좋아하신다니 좋네요! SF 영화는 상상력이 풍부하고 미래의 기술이나 우주 탐사와 같은 다양한 주제를 다루기 때문에 많은 사람들이 즐기는 장르입니다. 어떤 특정한 SF 영화가 궁금하신가요? 아니면 인기 있는 SF 영화를 추천해 드릴까요?
You: 인셉션이랑 인터스텔라는 이미 봤어
Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_gIL8moz52Q0u6zxQKcOWYkCm', function=Function(arguments='{}', name='get_popular_movies'), type='fun

BadRequestError: Error code: 400 - {'error': {'message': "Invalid value for 'content': expected a string, got null.", 'type': 'invalid_request_error', 'param': 'messages.[8].content', 'code': None}}

In [ ]:
str = '{"type": "function", "function": "name"}'

jsonObj = json.loads(str)
print(jsonObj)

def function_call(type, function):
    print(type, function)
function_call(**jsonObj)
